In [1]:
!rm -rf /kaggle/working/*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv")

In [4]:
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [5]:
train.shape

(439140, 16)

In [6]:
train.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [7]:
test.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [8]:
test.shape

(188165, 15)

In [9]:
test.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
dtype: int64

In [10]:
sub.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [11]:
train_ids = train['id'].values
test_ids = test['id'].values

df = pd.concat([train, test], axis=0, ignore_index=True)

df['Expected_Total_Laps'] = df['LapNumber'] / (df['RaceProgress'] + 1e-5)
df['Laps_Remaining'] = df['Expected_Total_Laps'] - df['LapNumber']
df['TyreLife_Ratio'] = df['TyreLife'] / (df['LapNumber'] + 1e-5)
df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
df['Pace_Drop'] = df['LapTime_Delta'] / (df['LapTime (s)'] + 1e-5)
df['Risk_Score'] = df['Cumulative_Degradation'] * df['TyreLife']
df['Stint_Progress'] = df['Stint'] * df['RaceProgress']
df['Laps_On_Tyre_Ratio'] = df['TyreLife'] / (df['Expected_Total_Laps'] + 1e-5)
df['Degradation_Per_Lap'] = df['Cumulative_Degradation'] / (df['LapNumber'] + 1e-5)
df['LapTime_x_Degradation'] = df['LapTime (s)'] * df['Cumulative_Degradation']
df['Position_x_Degradation'] = df['Position'] * df['Cumulative_Degradation']
df['TyreLife_Squared'] = df['TyreLife'] ** 2
df['Degradation_Squared'] = df['Cumulative_Degradation'] ** 2
df['RaceProgress_Squared'] = df['RaceProgress'] ** 2
df['Stint_x_TyreLife'] = df['Stint'] * df['TyreLife']
df['Position_x_RaceProgress'] = df['Position'] * df['RaceProgress']
df['Laps_Remaining_x_Degradation'] = df['Laps_Remaining'] * df['Cumulative_Degradation']
df['Late_Race_Flag'] = (df['RaceProgress'] > 0.75).astype(int)
df['Early_Race_Flag'] = (df['RaceProgress'] < 0.25).astype(int)
df['Tyre_Cliff_Risk'] = (df['TyreLife'] > 20).astype(int) * df['Degradation_Rate']
df['Optimal_Window'] = ((df['TyreLife'] > 15) & (df['TyreLife'] < 35)).astype(int)
df['Position_x_LapsRemaining'] = df['Position'] * df['Laps_Remaining']
df['Degradation_x_RaceProgress'] = df['Cumulative_Degradation'] * df['RaceProgress']
df['Compound_Race_AvgStintLen'] = df.groupby(['Compound','Race'])['TyreLife'].transform('mean')
df['TyreLife_vs_Avg'] = df['TyreLife'] - df['Compound_Race_AvgStintLen']
df['Near_Pit_Window'] = (df['TyreLife_vs_Avg'].abs() < 3).astype(int)
df['Position_Urgency'] = df['Position'] * (1 - df['RaceProgress'])
df['Deg_EWM'] = df.groupby(['Driver','Race','Year','Stint'])['Cumulative_Degradation'].transform(lambda x: x.ewm(span=3).mean())
df['LapTime_EWM'] = df.groupby(['Driver','Race','Year','Stint'])['LapTime (s)'].transform(lambda x: x.ewm(span=3).mean())
df['Compound_Race_AvgDegRate'] = df.groupby(['Compound','Race'])['Degradation_Rate'].transform('mean')
df['Relative_DegRate'] = df['Degradation_Rate'] / (df['Compound_Race_AvgDegRate'] + 1e-5)
df['Stint_x_Position'] = df['Stint'] * df['Position']
df['TyreLife_x_RaceProgress'] = df['TyreLife'] * df['RaceProgress']
df['Laps_Remaining_x_Position'] = df['Laps_Remaining'] * df['Position']

df['StintLen_So_Far'] = df.groupby(['Driver','Race','Year','Stint'])['TyreLife'].transform('max')
df['LapTime_vs_StintBest'] = df['LapTime (s)'] - df.groupby(['Driver','Race','Year','Stint'])['LapTime (s)'].transform('min')
df['TyreAge_vs_MedianPit'] = df['TyreLife'] - df.groupby(['Compound','Race'])['TyreLife'].transform('median')
df['Driver_Race_AvgLapTime'] = df.groupby(['Driver','Race','Year'])['LapTime (s)'].transform('mean')
df['LapTime_vs_DriverAvg'] = df['LapTime (s)'] - df['Driver_Race_AvgLapTime']
df['Compound_Race_MaxTyreLife'] = df.groupby(['Compound','Race'])['TyreLife'].transform('max')
df['TyreLife_PctOfMax'] = df['TyreLife'] / (df['Compound_Race_MaxTyreLife'] + 1e-5)
df['Deg_x_LapsRemaining'] = df['Degradation_Rate'] * df['Laps_Remaining']
df['Position_Change_Urgency'] = df['Position'] * df['Degradation_Rate'] * (1 - df['RaceProgress'])
df['Stint_TyreLife_Position'] = df['Stint'] * df['TyreLife'] * df['Position']

df['TyreLife_x_DegRate'] = df['TyreLife'] * df['Degradation_Rate']
df['Pit_Urgency'] = df['Degradation_Rate'] * df['Laps_Remaining'] * df['Position']
df['LapTime_x_RaceProgress'] = df['LapTime (s)'] * df['RaceProgress']
df['Compound_TyreLife_Rank'] = df.groupby(['Compound','Race','Year'])['TyreLife'].rank(pct=True)
df['Driver_Race_PitCount'] = df.groupby(['Driver','Race','Year'])['PitStop'].transform('sum')
df['Stints_Remaining_Est'] = (2 - df['Stint']).clip(lower=0)
df['Laps_Until_Typical_Pit'] = df['Compound_Race_AvgStintLen'] - df['TyreLife']

rank_cols = ['TyreLife', 'Cumulative_Degradation', 'LapTime (s)', 'Degradation_Rate', 'Risk_Score']
for col in rank_cols:
    df[f'{col}_rank'] = df.groupby(['Race', 'Year', 'LapNumber'])[col].rank(pct=True)

def add_lag_features(subset):
    subset = subset.sort_values(['Driver', 'Race', 'Year', 'LapNumber']).copy()
    grp = subset.groupby(['Driver', 'Race', 'Year', 'Stint'])
    subset['LapTime_Lag1'] = grp['LapTime (s)'].shift(1)
    subset['LapTime_Lag2'] = grp['LapTime (s)'].shift(2)
    subset['Degradation_Lag1'] = grp['Cumulative_Degradation'].shift(1)
    subset['LapTime_Rolling_Mean_3'] = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    subset['LapTime_Rolling_Std_3'] = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
    subset['Degradation_Rolling_Mean_3'] = grp['Cumulative_Degradation'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    subset['LapTime_Rolling_Mean_5'] = grp['LapTime (s)'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    subset['Deg_Accel'] = grp['Cumulative_Degradation'].transform(lambda x: x.diff().diff().fillna(0))
    subset['LapTime_Diff'] = subset['LapTime (s)'] - subset['LapTime_Lag1']
    subset['TyreLife_Lag1'] = grp['TyreLife'].shift(1)
    subset['DegRate_Lag1'] = grp['Degradation_Rate'].shift(1)
    subset['DegRate_Rolling_Mean_3'] = grp['Degradation_Rate'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    subset['LapTime_Lag3'] = grp['LapTime (s)'].shift(3)
    subset['TyreLife_x_DegRate_Lag1'] = grp['TyreLife_x_DegRate'].shift(1)
    subset['Pit_Urgency_Lag1'] = grp['Pit_Urgency'].shift(1)
    subset['DegRate_Rolling_Std_3'] = grp['Degradation_Rate'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
    subset['LapTime_Rolling_Mean_7'] = grp['LapTime (s)'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    subset['Deg_Rolling_Max_3'] = grp['Cumulative_Degradation'].transform(lambda x: x.rolling(3, min_periods=1).max())
    subset[['LapTime_Lag1','LapTime_Lag2','LapTime_Lag3','Degradation_Lag1','LapTime_Diff',
            'TyreLife_Lag1','DegRate_Lag1','TyreLife_x_DegRate_Lag1','Pit_Urgency_Lag1']] = \
        subset[['LapTime_Lag1','LapTime_Lag2','LapTime_Lag3','Degradation_Lag1','LapTime_Diff',
                'TyreLife_Lag1','DegRate_Lag1','TyreLife_x_DegRate_Lag1','Pit_Urgency_Lag1']].fillna(0)
    return subset

train_fe = add_lag_features(df[df['id'].isin(train_ids)].copy())
test_fe  = add_lag_features(df[df['id'].isin(test_ids)].copy())

lag_cols = ['LapTime_Lag1','LapTime_Lag2','LapTime_Lag3','Degradation_Lag1',
            'LapTime_Rolling_Mean_3','LapTime_Rolling_Std_3',
            'Degradation_Rolling_Mean_3','LapTime_Diff',
            'LapTime_Rolling_Mean_5','LapTime_Rolling_Mean_7',
            'Deg_Accel','Deg_Rolling_Max_3',
            'TyreLife_Lag1','DegRate_Lag1','DegRate_Rolling_Mean_3','DegRate_Rolling_Std_3',
            'TyreLife_x_DegRate_Lag1','Pit_Urgency_Lag1']

train_fe_indexed = train_fe.set_index('id')
test_fe_indexed  = test_fe.set_index('id')

for col in lag_cols:
    df.loc[df['id'].isin(train_ids), col] = train_fe_indexed[col].reindex(
        df.loc[df['id'].isin(train_ids), 'id']).values
    df.loc[df['id'].isin(test_ids), col] = test_fe_indexed[col].reindex(
        df.loc[df['id'].isin(test_ids), 'id']).values

cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

train_df = df[df['PitNextLap'].notnull()].copy()
test_df  = df[df['PitNextLap'].isnull()].copy()

drop_cols = ['id', 'PitNextLap', 'Compound_Race_AvgStintLen', 'Compound_Race_AvgDegRate',
             'Driver_Race_AvgLapTime', 'Compound_Race_MaxTyreLife']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X      = train_df[feature_cols].reset_index(drop=True)
y      = train_df['PitNextLap'].reset_index(drop=True)
X_test = test_df[feature_cols].reset_index(drop=True)

print(f"Features: {X.shape[1]} | Train: {X.shape[0]} | Test: {X_test.shape[0]}")
print(f"Target mean: {y.mean():.4f}")

Features: 84 | Train: 439140 | Test: 188165
Target mean: 0.1990


In [12]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata

N_SPLITS = 5
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

lgb_params = {
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'num_leaves': 63,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'min_split_gain': 0.01,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'gpu',
    'gpu_use_dp': False,
    'verbose': -1,
}

xgb_params = {
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'gamma': 0.05,
    'random_state': 42,
    'eval_metric': 'auc',
    'n_jobs': -1,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 100,
}

cb_params = {
    'iterations': 3000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 5,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'min_data_in_leaf': 20,
    'random_state': 42,
    'eval_metric': 'AUC',
    'verbose': False,
    'task_type': 'GPU',
    'devices': '0',
    'early_stopping_rounds': 100,
}

et_params = {
    'n_estimators': 150,
    'max_depth': 10,
    'min_samples_leaf': 10,
    'max_features': 0.75,
    'random_state': 42,
    'n_jobs': -1,
}

mlp_params = {
    'hidden_layer_sizes': (256, 128, 64),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.01,
    'batch_size': 2048,
    'learning_rate_init': 0.001,
    'max_iter': 100,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 15,
    'random_state': 42,
}

lgb_test_preds = np.zeros(len(X_test))
xgb_test_preds = np.zeros(len(X_test))
cb_test_preds  = np.zeros(len(X_test))
et_test_preds  = np.zeros(len(X_test))
mlp_test_preds = np.zeros(len(X_test))
lgb_oof = np.zeros(len(X))
xgb_oof = np.zeros(len(X))
cb_oof  = np.zeros(len(X))
et_oof  = np.zeros(len(X))
mlp_oof = np.zeros(len(X))

scaler = StandardScaler()
X_scaled      = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]

    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )
    lgb_oof[val_idx]  = model_lgb.predict_proba(X_val)[:, 1]
    lgb_test_preds   += model_lgb.predict_proba(X_test)[:, 1] / N_SPLITS

    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx]  = model_xgb.predict_proba(X_val)[:, 1]
    xgb_test_preds   += model_xgb.predict_proba(X_test)[:, 1] / N_SPLITS

    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100)
    cb_oof[val_idx]   = model_cb.predict_proba(X_val)[:, 1]
    cb_test_preds    += model_cb.predict_proba(X_test)[:, 1] / N_SPLITS

    model_et = ExtraTreesClassifier(**et_params)
    model_et.fit(X_train, y_train)
    et_oof[val_idx]   = model_et.predict_proba(X_val)[:, 1]
    et_test_preds    += model_et.predict_proba(X_test)[:, 1] / N_SPLITS

    model_mlp = MLPClassifier(**mlp_params)
    model_mlp.fit(X_scaled[train_idx], y.iloc[train_idx])
    mlp_oof[val_idx]  = model_mlp.predict_proba(X_scaled[val_idx])[:, 1]
    mlp_test_preds   += model_mlp.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

    print(f"Fold {fold+1} | LGB: {roc_auc_score(y_val, lgb_oof[val_idx]):.5f} | "
          f"XGB: {roc_auc_score(y_val, xgb_oof[val_idx]):.5f} | "
          f"CB: {roc_auc_score(y_val, cb_oof[val_idx]):.5f} | "
          f"ET: {roc_auc_score(y_val, et_oof[val_idx]):.5f} | "
          f"MLP: {roc_auc_score(y_val, mlp_oof[val_idx]):.5f}")

print(f"\nOOF LGB: {roc_auc_score(y, lgb_oof):.5f}")
print(f"OOF XGB: {roc_auc_score(y, xgb_oof):.5f}")
print(f"OOF CB:  {roc_auc_score(y, cb_oof):.5f}")
print(f"OOF ET:  {roc_auc_score(y, et_oof):.5f}")
print(f"OOF MLP: {roc_auc_score(y, mlp_oof):.5f}")

def rank_avg(*arrays):
    ranks = [rankdata(a) / len(a) for a in arrays]
    return np.mean(ranks, axis=0)

et_oof_score  = roc_auc_score(y, et_oof)
mlp_oof_score = roc_auc_score(y, mlp_oof)

base_oofs  = [lgb_oof, xgb_oof, cb_oof]
base_preds = [lgb_test_preds, xgb_test_preds, cb_test_preds]

if et_oof_score >= 0.90:
    base_oofs.append(et_oof)
    base_preds.append(et_test_preds)
    print(f"ET included ({et_oof_score:.5f})")
else:
    print(f"ET excluded ({et_oof_score:.5f})")

if mlp_oof_score >= 0.90:
    base_oofs.append(mlp_oof)
    base_preds.append(mlp_test_preds)
    print(f"MLP included ({mlp_oof_score:.5f})")
else:
    print(f"MLP excluded ({mlp_oof_score:.5f})")

simple_oof = np.mean(base_oofs, axis=0)
simple_avg = np.mean(base_preds, axis=0)

rank_oof   = rank_avg(*base_oofs)
rank_preds = rank_avg(*base_preds)

oof_stack  = np.column_stack(base_oofs)
test_stack = np.column_stack(base_preds)

meta_lgb = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.01, num_leaves=15,
    min_child_samples=30, verbose=-1, random_state=42,
    reg_alpha=0.1, reg_lambda=1.0
)
kf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
stacked_oof  = np.zeros(len(X))
stacked_test = np.zeros(len(X_test))
for tr_idx, vl_idx in kf_meta.split(oof_stack, y):
    meta_lgb.fit(oof_stack[tr_idx], y.iloc[tr_idx])
    stacked_oof[vl_idx] = meta_lgb.predict_proba(oof_stack[vl_idx])[:, 1]
    stacked_test += meta_lgb.predict_proba(test_stack)[:, 1] / 5

print(f"OOF Simple Avg:  {roc_auc_score(y, simple_oof):.5f}")
print(f"OOF Rank Avg:    {roc_auc_score(y, rank_oof):.5f}")
print(f"OOF Stacked:     {roc_auc_score(y, stacked_oof):.5f}")

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1 | LGB: 0.94970 | XGB: 0.94979 | CB: 0.94950 | ET: 0.93455 | MLP: 0.93842


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2 | LGB: 0.94796 | XGB: 0.94807 | CB: 0.94754 | ET: 0.93272 | MLP: 0.93548


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3 | LGB: 0.94928 | XGB: 0.94933 | CB: 0.94897 | ET: 0.93465 | MLP: 0.93787


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4 | LGB: 0.94801 | XGB: 0.94784 | CB: 0.94770 | ET: 0.93269 | MLP: 0.93748


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5 | LGB: 0.94939 | XGB: 0.94933 | CB: 0.94884 | ET: 0.93400 | MLP: 0.93764

OOF LGB: 0.94887
OOF XGB: 0.94887
OOF CB:  0.94850
OOF ET:  0.93369
OOF MLP: 0.93718
ET included (0.93369)
MLP included (0.93718)
OOF Simple Avg:  0.94805
OOF Rank Avg:    0.94768
OOF Stacked:     0.94962


In [13]:
if roc_auc_score(y, stacked_oof) > roc_auc_score(y, simple_oof):
    final_preds = stacked_test
    print("Using stacked predictions")
else:
    final_preds = simple_avg
    print("Using simple average predictions")

print(f"Final preds mean: {final_preds.mean():.4f} | Target mean: {y.mean():.4f}")

assert abs(final_preds.mean() - y.mean()) < 0.15, "WARNING: prediction mean too far from target mean — check for flip!"

sub['PitNextLap'] = final_preds
sub.to_csv('submission.csv', index=False)
sub.head()

Using stacked predictions
Final preds mean: 0.1977 | Target mean: 0.1990


,id,PitNextLap
0,439140,0.004853
1,439141,0.005502
2,439142,0.002714
3,439143,0.137922
4,439144,0.859636
